# Exact attribution and structural evaluation

In [ ]:
from pathlib import Path
import gzip
import sys
import tarfile

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, to_hex
from matplotlib.patches import PathPatch
from matplotlib.textpath import TextPath
from matplotlib.transforms import Affine2D
import numpy as np
import pandas as pd
import py3Dmol
import torch
from IPython.display import display
from scipy.stats import spearmanr

# ---------------------------------------------------------------------
# Paths and run settings
# ---------------------------------------------------------------------
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "lamina.py").is_file():
    raise RuntimeError("Start this notebook from the LAMINA repository root")
DATA = ROOT / "Data" / "NetMHCpan"
PDB_ROOT = ROOT / "Data" / "PDB"
BA_CHECKPOINT = ROOT / "final_models" / "lamina_ba.pt"
STRUCTURES = ROOT / "artifacts" / "pmhc_mhci_distance_matrices.jsonl"
FOLDX_RESULTS = ROOT / "artifacts" / "foldx_peptide_scan.csv"
OUT = ROOT / "artifacts" / "interpretability"
SASA_RESULTS = ROOT / "artifacts" / "peptide_sasa.jsonl"
IEDB_MATRICES = (
    ROOT
    / "Data"
    / "reference_motifs"
    / "IEDB_MHC_I-2.9_matx_smm_smmpmbec.tar.gz"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
LOGO_ALLELE = "HLA-C*02:02"
MOTIF_ALLELES = (
    "HLA-A*01:01",
    "HLA-A*02:01",
    "HLA-A*03:01",
    "HLA-A*24:02",
    "HLA-B*07:02",
    "HLA-B*27:05",
)
DISPLAY_PDBS = ("3P9M", "7K80")

sys.path.insert(0, str(ROOT / "src"))
from lamina import CANONICAL_AA, encode_batch, load_model
from attribution import ResidueAttribution
from pmhc_mhci_common import path_for_pdb_id

OUT.mkdir(parents=True, exist_ok=True)
model = load_model(BA_CHECKPOINT, DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"device={DEVICE}; model parameters={parameter_count:,}")


## Exact residue attribution

Each convolution is bias-free and linear. We therefore expand every
motif window back onto the residues that produced it, contract the HLA
and peptide components with the single learned interaction map, and
divide by the same number of valid motif pairs used by the prediction.
The resulting 34 by peptide-length matrix sums exactly to the logit.

The notebook and structural runner share the implementation in `src/attribution.py`.


In [ ]:
attribution = ResidueAttribution(model)

example_hla = "YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY"
example_peptide = "GILGFVFTL"
example_pair_map, example_contributions, _ = attribution(example_hla, example_peptide)
print(f"Exact 34 x {len(example_peptide)} attribution; logit={example_pair_map.sum():.6f}")


## Alleles and logo rendering

Load the NetMHCpan pseudosequences and define the letter renderer used by the exact logos below.


In [ ]:
def normalize_allele(value):
    value = value.upper().replace("HLA-", "").replace("*", "").replace(":", "")
    return value


pseudosequence_table = {}
with (DATA / "NetMHCpan_train" / "pseudoseqs").open() as handle:
    for line in handle:
        fields = line.split()
        if len(fields) >= 2:
            pseudosequence_table[normalize_allele(fields[0])] = fields[1].upper()
logo_pseudosequence = pseudosequence_table[normalize_allele(LOGO_ALLELE)]

amino_acid_colors = {
    **dict.fromkeys("AVILMFWY", "#2f7d32"),
    **dict.fromkeys("STNQ", "#1976d2"),
    **dict.fromkeys("KRH", "#c62828"),
    **dict.fromkeys("DE", "#6a1b9a"),
    **dict.fromkeys("CGP", "#8e6bb5"),
}


def draw_letter(axis, letter, x, y, height):
    '''Scale a matplotlib text outline to one logo-letter rectangle.'''
    path = TextPath((0, 0), letter, size=1, prop={"weight": "bold"})
    bounds = path.get_extents()
    transform = (
        Affine2D()
        .scale(0.86 / bounds.width, abs(height) / bounds.height)
        .translate(x - 0.43, y)
        + axis.transData
    )
    axis.add_patch(
        PathPatch(path, transform=transform, color=amino_acid_colors[letter], lw=0)
    )



## Exact centered-logit BA sequence logo

For a fixed HLA and peptide length, LAMINA's raw logit is additive in
the peptide residues:

$$z(a,b)=c_{a,L}+\sum_{j=1}^{L}\beta_{j,b_j}.$$

Start from any background peptide and make every amino-acid substitution
at every position. For fixed position $j$, its 20 logits have the form
$z_{j,c}=C_j+\beta_{j,c}$. Row-centering therefore removes the arbitrary
background exactly:

$$\Delta_{j,c}=z_{j,c}-\frac{1}{20}\sum_a z_{j,a}.$$

Thus one batch of $20L$ raw-logit evaluations gives the exact signed logo.
The best residue can also be chosen independently at every position, so
$b_j^*=\arg\max_c\Delta_{j,c}$ is the global maximizing peptide. No
sigmoid is applied below. All 20 scores are retained; the plot shows the
eight largest absolute effects at each position for readability.


In [ ]:
@torch.inference_mode()
def exact_logit_logo(ba_model, pseudosequence, peptide_length=9):
    '''Return the exact centered logo and globally maximizing peptide.'''
    device = ba_model.token_embedding.weight.device
    background = "A" * peptide_length
    substitutions = []
    for position in range(peptide_length):
        for amino_acid in CANONICAL_AA:
            peptide = list(background)
            peptide[position] = amino_acid
            substitutions.append("".join(peptide))

    hla_ids, _ = encode_batch([pseudosequence] * len(substitutions), 34, device)
    peptide_ids, lengths = encode_batch(substitutions, peptide_length, device)
    logits, _ = ba_model(hla_ids, peptide_ids, lengths)
    substitution_logits = logits.reshape(peptide_length, len(CANONICAL_AA))
    centered_logits = substitution_logits - substitution_logits.mean(
        dim=1, keepdim=True
    )

    best_indices = centered_logits.argmax(dim=1)
    best_peptide = "".join(
        CANONICAL_AA[index] for index in best_indices.tolist()
    )

    # Check the additive reconstruction against a direct model evaluation.
    background_index = CANONICAL_AA.index("A")
    positions = torch.arange(peptide_length, device=substitution_logits.device)
    reconstructed_logit = substitution_logits[0, background_index] + (
        substitution_logits[positions, best_indices]
        - substitution_logits[:, background_index]
    ).sum()
    best_hla_ids, _ = encode_batch([pseudosequence], 34, device)
    best_peptide_ids, best_lengths = encode_batch([best_peptide], peptide_length, device)
    direct_logit, _ = ba_model(best_hla_ids, best_peptide_ids, best_lengths)
    torch.testing.assert_close(
        reconstructed_logit, direct_logit[0], atol=2e-5, rtol=2e-5
    )

    return centered_logits.cpu().numpy(), best_peptide, direct_logit.item()


exact_ba_logo, best_ba_peptide, best_ba_logit = exact_logit_logo(
    model, logo_pseudosequence, peptide_length=9
)
display(pd.DataFrame([{
    "task": "BA",
    "maximizing peptide": best_ba_peptide,
    "raw logit": best_ba_logit,
}]).set_index("task"))


In [ ]:
def plot_signed_logo(axis, logo):
    positive_tops, negative_bottoms = [], []
    for position, values in enumerate(logo):
        selected = np.argsort(np.abs(values))[-8:]
        positive_y = 0.0
        positive = selected[values[selected] > 0]
        for index in sorted(positive, key=lambda i: values[i]):
            draw_letter(
                axis, CANONICAL_AA[index], position, positive_y, values[index]
            )
            positive_y += values[index]

        negative_y = 0.0
        negative = selected[values[selected] < 0]
        for index in sorted(negative, key=lambda i: abs(values[i])):
            negative_y += values[index]
            draw_letter(
                axis, CANONICAL_AA[index], position, negative_y, values[index]
            )

        positive_tops.append(positive_y)
        negative_bottoms.append(negative_y)

    axis.axhline(0, color="black", lw=0.8)
    axis.set(
        xlim=(-0.6, logo.shape[0] - 0.4),
        ylim=(min(negative_bottoms) * 1.08, max(positive_tops) * 1.08),
    )
    axis.set_xticks(range(logo.shape[0]), range(1, logo.shape[0] + 1))
    axis.spines[["top", "right"]].set_visible(False)


figure, axis = plt.subplots(figsize=(5, 4.5))
plot_signed_logo(axis, exact_ba_logo)
axis.set(
    xlabel="Peptide position",
    ylabel="BA centered raw logit",
)
figure.suptitle(f"Exact BA 9-mer logo for {LOGO_ALLELE}")
figure.subplots_adjust(left=0.10, right=0.98, bottom=0.12, top=0.88)
plt.show()

pd.DataFrame(exact_ba_logo, columns=list(CANONICAL_AA), index=np.arange(1, 10)).to_csv(OUT / "exact_ba_logo.csv", index_label="position")


## Exact BA preferences versus IEDB SMM

We compare six common alleles with the position-specific
[IEDB SMM matrices](https://tools.iedb.org/mhci/download/). SMM predicts
$\log_{10}(\mathrm{IC}_{50})$, so its weights are negated to make larger
values mean stronger binding.

For a common scale, the 20 amino-acid values are standardized separately
within every allele and peptide position. The pooled Spearman correlation
below measures descriptive motif concordance; it is not a held-out
performance estimate.


In [ ]:
def position_zscore(values):
    '''Standardize the 20 amino-acid values within each position.'''
    values = np.asarray(values, dtype=float)
    scale = values.std(axis=1, keepdims=True)
    if np.any(scale == 0):
        raise ValueError("A motif position has zero variance")
    return (values - values.mean(axis=1, keepdims=True)) / scale


def read_iedb_smm(archive, allele, peptide_length=9):
    '''Read one position by amino-acid SMM matrix from the IEDB archive.'''
    member = f"smm_matrix/{allele.replace('*', '-')}-{peptide_length}.txt"
    lines = archive.extractfile(member).read().decode().splitlines()
    if int(lines[0].split()[1]) != peptide_length:
        raise ValueError(f"Unexpected peptide length in {member}")

    weights = {}
    for line in lines[1:]:
        fields = line.split()
        if fields and fields[0] in CANONICAL_AA:
            weights[fields[0]] = [float(value) for value in fields[1:]]
    if set(weights) != set(CANONICAL_AA):
        raise ValueError(f"Incomplete amino-acid matrix in {member}")
    return np.asarray([weights[aa] for aa in CANONICAL_AA]).T


with tarfile.open(IEDB_MATRICES, "r:gz") as archive:
    smm_logo_z = {
        allele: position_zscore(-read_iedb_smm(archive, allele))
        for allele in MOTIF_ALLELES
    }


In [ ]:
lamina_ba_logo_z = {}
comparison_frames = []
for allele in MOTIF_ALLELES:
    pseudosequence = pseudosequence_table[normalize_allele(allele)]
    centered_logo, _, _ = exact_logit_logo(
        model, pseudosequence, peptide_length=9
    )
    lamina_ba_logo_z[allele] = position_zscore(centered_logo)
    comparison_frames.append(pd.DataFrame({
        "allele": allele,
        "position": np.repeat(np.arange(1, 10), len(CANONICAL_AA)),
        "amino_acid": np.tile(list(CANONICAL_AA), 9),
        "smm_z": smm_logo_z[allele].ravel(),
        "lamina_ba_z": lamina_ba_logo_z[allele].ravel(),
    }))
known_motif_comparison = pd.concat(comparison_frames, ignore_index=True)

motif_agreement = pd.DataFrame([{
    "task": "BA",
    "reference": "IEDB SMM",
    "points": len(known_motif_comparison),
    "pooled Spearman rho": spearmanr(
        known_motif_comparison.smm_z,
        known_motif_comparison.lamina_ba_z,
    ).statistic,
}]).set_index("task")
display(motif_agreement)

known_motif_comparison.to_csv(OUT / "motif_comparison.csv", index=False)
motif_agreement.to_csv(OUT / "motif_agreement.csv")


In [ ]:
logo_columns = (
    ("Exact BA", lamina_ba_logo_z),
    ("IEDB SMM", smm_logo_z),
)
figure, axes = plt.subplots(len(MOTIF_ALLELES), 2, figsize=(9, 13), sharex=True)
for row, allele in enumerate(MOTIF_ALLELES):
    for column, (title, logos) in enumerate(logo_columns):
        axis = axes[row, column]
        plot_signed_logo(axis, logos[allele])
        axis.set_yticks([])
        axis.spines["left"].set_visible(False)
        if row == 0:
            axis.set_title(title)
        if row < len(MOTIF_ALLELES) - 1:
            axis.tick_params(labelbottom=False)
        if column == 0:
            axis.set_ylabel(allele, rotation=0, ha="right", va="center")
for axis in axes[-1]:
    axis.set_xlabel("Peptide position")
figure.suptitle("Position-standardized BA 9-mer preference logos")
figure.subplots_adjust(
    left=0.16, right=0.99, bottom=0.055, top=0.94, hspace=0.12, wspace=0.08
)
plt.show()


In [ ]:
# Colors group alleles by locus; marker shapes distinguish them in grayscale.
motif_colors = {
    "HLA-A*01:01": "#173F5F",
    "HLA-A*02:01": "#3F6F8F",
    "HLA-A*03:01": "#6F96AF",
    "HLA-A*24:02": "#9EB8C9",
    "HLA-B*07:02": "#934A3B",
    "HLA-B*27:05": "#CB7B5B",
}
motif_markers = dict(zip(MOTIF_ALLELES, ("o", "s", "^", "D", "P", "v")))
nature_style = {
    "font.family": "Arial",
    "font.size": 6.5,
    "axes.labelsize": 7,
    "axes.linewidth": 0.65,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "xtick.major.size": 2.8,
    "ytick.major.size": 2.8,
    "xtick.major.width": 0.65,
    "ytick.major.width": 0.65,
    "legend.fontsize": 5.8,
}

lower = min(
    known_motif_comparison.smm_z.min(),
    known_motif_comparison.lamina_ba_z.min(),
)
upper = max(
    known_motif_comparison.smm_z.max(),
    known_motif_comparison.lamina_ba_z.max(),
)
padding = 0.035 * (upper - lower)
lower, upper = lower - padding, upper + padding
rho = spearmanr(
    known_motif_comparison.smm_z,
    known_motif_comparison.lamina_ba_z,
).statistic
with plt.rc_context(nature_style):
    figure, axis = plt.subplots(figsize=(4.72, 3.35))  # 120 mm wide
    for allele in MOTIF_ALLELES:
        points = known_motif_comparison[
            known_motif_comparison.allele.eq(allele)
        ]
        axis.scatter(
            points.smm_z,
            points.lamina_ba_z,
            s=7.5,
            alpha=0.42,
            color=motif_colors[allele],
            marker=motif_markers[allele],
            linewidths=0,
            label=allele,
            zorder=2,
        )

    axis.plot(
        [lower, upper],
        [lower, upper],
        color="#A8A8A8",
        linestyle=(0, (3, 2)),
        linewidth=0.65,
        zorder=1,
    )
    axis.set(
        xlim=(lower, upper),
        ylim=(lower, upper),
        xticks=np.arange(-3, 4),
        yticks=np.arange(-3, 4),
        xlabel="IEDB SMM preference",
        ylabel="LAMINA Exact preference",
    )
    axis.set_aspect("equal", adjustable="box")
    axis.text(
        0.03,
        0.97,
        rf"$\rho$ = {rho:.3f}"
        f"\n$n$ = {len(known_motif_comparison):,}",
        transform=axis.transAxes,
        ha="left",
        va="top",
        fontsize=6.3,
        linespacing=1.25,
    )
    axis.tick_params(direction="out", top=False, right=False, pad=2)
    axis.spines[["top", "right"]].set_visible(False)

    legend = axis.legend(
        loc="center left",
        bbox_to_anchor=(1.015, 0.5),
        frameon=False,
        ncol=1,
        borderaxespad=0,
        handletextpad=0.45,
        labelspacing=0.45,
        markerscale=1.35,
    )
    for handle in legend.legend_handles:
        handle.set_alpha(1.0)
    figure.subplots_adjust(left=0.16, right=0.72, bottom=0.18, top=0.98)
    plt.show()


## Structural cohort and exact peptide contributions

The structural analysis uses deposited-sequence alignment and exact chain/residue
coordinates. SASA retains modified polymer atoms in both bound and free peptide
geometries. Residues without a supported maximum ASA remain in raw buried-SASA
correlations and are excluded only from normalized-SASA classification.

The cell below runs `src/structural_analysis.py`, which verifies the structural,
SASA, and FoldX input provenance before joining residue energies. First generate
these inputs with the commands in the README. Output tables and the run
manifest are written to `artifacts/interpretability`.

The same analysis can run without the logo cells:

```bash
python src/structural_analysis.py \
  --structures artifacts/pmhc_mhci_distance_matrices.jsonl \
  --checkpoint final_models/lamina_ba.pt \
  --sasa artifacts/peptide_sasa.jsonl \
  --foldx-results artifacts/foldx_peptide_scan.csv \
  --pdb-root Data/PDB \
  --output artifacts/interpretability
```


In [ ]:
# Validate structure, SASA, and FoldX inputs before joining residue results.
from structural_analysis import run_analysis

structural_result = run_analysis(
    structures=STRUCTURES,
    checkpoint=BA_CHECKPOINT,
    sasa_path=SASA_RESULTS,
    pdb_root=PDB_ROOT,
    output=OUT,
    foldx_results=FOLDX_RESULTS,
)
records = structural_result["records"]
attribution_by_line = structural_result["attribution_by_line"]
foldx_df = structural_result["foldx_df"]
sasa_df = structural_result["sasa_df"]
summary = structural_result["summary"]
print(f"Structural cohort: {len(records):,} complexes")
print(f"Tables and provenance: {OUT}")


## FoldX peptide single mutants

Positive mutation energy means a weaker interface. The runner applies the
wild-type/interface quality filters to the manifest-verified local peptide scan and
joins each retained energy to its exact residue contribution.
The statistics and plot below show the results.


In [ ]:
foldx = foldx_df
foldx_statistics = pd.Series({
    "mutants": summary["foldx_residues"],
    "complexes": summary["foldx_complexes"],
    "pooled Spearman rho": summary["foldx_spearman"],
})
display(foldx_statistics.to_frame("value"))
display(structural_result["figures"]["peptide_foldx_single"])
print(f"FoldX input manifest: {FOLDX_RESULTS.with_suffix('.manifest.json')}")


## Peptide solvent-accessible surface area

SASA contributions are standardized within each complex. Raw buried SASA is
free-peptide minus bound exposure, with the runner's documented clipping.
Relative bound SASA below 0.5 defines the binary target. Unsupported modified
residue normalization is recorded explicitly and excluded from binary metrics;
finite raw buried SASA from those sites remains in the Spearman analysis.


In [ ]:
# Modified polymer atoms remain in both SASA states. Unsupported residue
# normalization excludes a site from binary/AUROC analysis only; its finite
# raw buried SASA remains in the correlation analysis.
sasa = sasa_df
sasa_statistics = pd.Series({
    "raw SASA residues": summary["sasa_residues"],
    "complexes": summary["sasa_complexes"],
    "pooled buried-SASA Spearman rho": summary["buried_sasa_pooled_spearman"],
    "mean per-complex Spearman rho": summary["buried_sasa_mean_individual_spearman"],
    "normalized SASA residues": summary["normalized_sasa_residues"],
    "normalization exclusions": summary["normalization_excluded_residues"],
    "pooled AUROC": summary["pooled_auc"],
    "AUROC-eligible complexes": summary["individual_auc_complexes"],
    "mean per-complex AUROC": summary["mean_individual_auc"],
    "median per-complex AUROC": summary["median_individual_auc"],
})
display(sasa_statistics.to_frame("value"))
display(structural_result["figures"]["sasa_buried_scatter"])

# Both binary plots use only sites with supported, finite normalization.
display(structural_result["figures"]["sasa_pooled_roc"])
display(structural_result["figures"]["sasa_individual_auc"])
print(f"Summary: {OUT / 'structural_summary.json'}")
print(f"Input/output provenance: {OUT / 'run_manifest.json'}")


## Structural projections

Peptide residues are colored by their own exact contribution. Each HLA
pseudosequence residue copies the contribution of its nearest peptide
residue; those HLA colors are a spatial projection, not a second HLA
attribution. The default examples are PDB 3P9M and 7K80; change `DISPLAY_PDBS` to inspect other entries in your cohort.


In [ ]:
def author_chain(mapping, fallback):
    return next(
        (row["auth_asym_id"] for row in mapping if row.get("auth_asym_id")),
        fallback,
    )


color_scale = plt.get_cmap("coolwarm")
display_records = []
for pdb_id in DISPLAY_PDBS:
    record = next((record for record in records if record["pdb_id"].upper() == pdb_id.upper()), None)
    if record is None:
        print(f"{pdb_id} is absent from this cohort; choose an entry in DISPLAY_PDBS")
    else:
        display_records.append(record)
if display_records:
    all_display_values = np.concatenate([
        attribution_by_line[record["line_index"]] for record in display_records
    ])
    color_limit = max(float(np.max(np.abs(all_display_values))), 1e-12)
    normalization = Normalize(-color_limit, color_limit)

for record in display_records:
    pdb_id = record["pdb_id"]
    contribution = attribution_by_line[record["line_index"]]
    matrix = np.asarray(record["distance_min_heavy_atom"], dtype=float)
    mhc_chain = author_chain(
        record["pseudosequence_residue_mappings"], record["mhc"]["asym_id"]
    )
    peptide_chain = author_chain(
        record["peptide_residue_mappings"], record["peptide_chain"]["asym_id"]
    )
    path = path_for_pdb_id(PDB_ROOT, pdb_id)
    opener = gzip.open if path.name.endswith(".gz") else open
    with opener(path, "rt") as handle:
        structure_text = handle.read()

    view = py3Dmol.view(width=650, height=450)
    view.addModel(structure_text, "cif")
    view.setStyle({}, {"cartoon": {"color": "#d1d5db", "opacity": 0.65}})

    # Color the peptide with its exact one-dimensional attribution.
    for mapping in record["peptide_residue_mappings"]:
        index = int(mapping["peptide_index"])
        residue_number = mapping.get("auth_seq_id") or mapping["structure_seq_id"]
        color = to_hex(color_scale(normalization(contribution[index])))
        view.setStyle(
            {"chain": peptide_chain, "resi": str(residue_number)},
            {"cartoon": {"color": color}, "stick": {"color": color, "radius": 0.22}},
        )

    # Project the nearest peptide value onto each of the 34 HLA residues.
    for pseudo_index, mapping in enumerate(record["pseudosequence_residue_mappings"]):
        distances = matrix[pseudo_index]
        nearest = int(np.nanargmin(distances))
        residue_number = mapping.get("auth_seq_id") or mapping["structure_seq_id"]
        color = to_hex(color_scale(normalization(contribution[nearest])))
        view.setStyle(
            {"chain": mhc_chain, "resi": str(residue_number)},
            {"cartoon": {"color": color}, "stick": {"color": color, "radius": 0.12}},
        )

    view.setBackgroundColor("white")
    view.zoomTo({"chain": peptide_chain})
    print(f"{pdb_id}: {record['peptide']}")
    view.show()
